# 19.4 因果图与 do-演算 / Causal DAGs & do-calculus

**中文**：19.1 说过,能随机化就用 A/B 测试。但很多时候**你无法随机化**——你只有一堆观测数据(如"喝咖啡的人是否更长寿"),不能强迫一半人喝咖啡。这时要从观测数据里推因果,就必须先想清楚:**变量之间的因果结构长什么样?哪些变量该控制(放进回归),哪些绝对不能控制?** 这套语言就是 **因果图(DAG, 有向无环图)** 与 Judea Pearl 的 **do-演算**。本节是整个因果推断的**思维地基**——搞懂它,才知道 19.5–19.8 那些方法到底在"控制"什么。
**English**: 19.1 said: if you can randomize, use an A/B test. But often **you can't randomize** — you only have observational data (e.g. "do coffee drinkers live longer"), and you can't force half the people to drink coffee. To infer causation from observational data, you must first clarify: **what is the causal structure among variables? which should be controlled (put in the regression), which must never be?** That language is the **causal DAG (Directed Acyclic Graph)** and Judea Pearl's **do-calculus**. This section is the **conceptual foundation** of all causal inference — understand it, and you'll know what the methods in 19.5–19.8 are actually "controlling for."

---

**中文**：**DAG** 用节点表示变量、箭头表示"直接因果"($A\to B$ 读作"A 直接导致 B")。整个因果推断的关键,是识别三种**基本三节点结构**——它们决定了"该不该控制中间那个变量":
**English**: A **DAG** uses nodes for variables and arrows for "direct causation" ($A\to B$ reads "A directly causes B"). The key to all causal inference is recognizing three **fundamental three-node structures** — they determine "whether to control the middle variable":

**中文**：
- **① 混杂/分叉(Confounder / Fork)**:$X\leftarrow Z\to Y$。$Z$ 是 $X$ 和 $Y$ 的**共同原因**。它制造 $X,Y$ 之间的**虚假相关**(明明 $X$ 不影响 $Y$,却看起来相关)。**必须控制 $Z$**(条件在它上面),才能得到真实因果效应。经典例子:冰淇淋销量与溺水人数相关——共同原因是"夏天/气温"。
- **② 中介/链(Mediator / Chain)**:$X\to M\to Y$。$M$ 在因果路径**上**($X$ 通过 $M$ 影响 $Y$)。若你想要 $X$ 对 $Y$ 的**总效应**,**绝不能控制 $M$**——控制它会**堵住**这条因果路,把真实效应抹成 0。例子:锻炼→降体重→降血压,若控制体重就看不到锻炼的总效应。
- **③ 对撞/碰撞(Collider)**:$X\to C\leftarrow Y$。$C$ 是 $X$ 和 $Y$ 的**共同结果**。此时 $X,Y$ 本来**独立**,但**一旦控制 $C$,反而会凭空制造出虚假相关**(对撞偏差/选择偏差)。**绝不能控制对撞变量**。这是最反直觉的一条。

**English**:
- **① Confounder / Fork**: $X\leftarrow Z\to Y$. $Z$ is a **common cause** of $X$ and $Y$. It creates **spurious correlation** between them (they look related even if $X$ doesn't affect $Y$). **You must control $Z$** (condition on it) to get the true causal effect. Classic example: ice-cream sales correlate with drownings — the common cause is "summer/temperature."
- **② Mediator / Chain**: $X\to M\to Y$. $M$ is **on** the causal path ($X$ affects $Y$ through $M$). For the **total effect** of $X$ on $Y$, **never control $M$** — doing so **blocks** the causal path, erasing the true effect to 0. Example: exercise → lower weight → lower blood pressure; controlling for weight hides exercise's total effect.
- **③ Collider**: $X\to C\leftarrow Y$. $C$ is a **common effect** of $X$ and $Y$. Here $X,Y$ are actually **independent**, but **once you control $C$, you conjure spurious correlation out of nothing** (collider bias / selection bias). **Never control a collider.** The most counterintuitive rule.

**中文**：所以"把所有变量都塞进回归"是**错的**。正确做法由**后门准则(backdoor criterion)** 给出:要估计 $X\to Y$,就控制一组变量**堵住所有"后门路径"(经由混杂的路径),同时不控制中介和对撞**。**do-演算**则给出符号:$P(Y|do(X))$(**干预** X 后 Y 的分布,即真因果)通常 $\ne P(Y|X)$(单纯**观察**到 X 时 Y 的分布,含混杂)。
**English**: So "throw all variables into the regression" is **wrong**. The right set is given by the **backdoor criterion**: to estimate $X\to Y$, control a set that **blocks all "backdoor paths" (paths through confounders) while not controlling mediators or colliders**. **do-calculus** gives the notation: $P(Y|do(X))$ (the distribution of Y after **intervening** on X, i.e. the true causal effect) generally $\ne P(Y|X)$ (the distribution of Y merely **observing** X, which includes confounding).

> 💡 **面试速查 / Interview cheat-sheet（★★★ 因果地基必考）**
> **中文**：DAG=因果结构图(节点=变量, 箭头=直接因)。三大结构决定"控不控中间变量":**混杂(X←Z→Y)必须控**(否则虚假相关)、**中介(X→M→Y)求总效应时别控**(会堵路)、**对撞(X→C←Y)绝不能控**(控了凭空造出虚假相关=选择偏差)。**后门准则**:控制变量堵住所有后门路径(经混杂), 不碰中介/对撞→得到无偏因果效应。**do 算子**:$P(Y|do(X))$(干预, 真因果) ≠ $P(Y|X)$(观察, 含混杂)。**核心教训**:"控制越多变量越准"是错的——控错(中介/对撞)会引入偏差。识别混杂需要**领域知识画 DAG**(数据本身分不清方向)。对撞例子:伯克森悖论、"幸存者偏差"、"为什么名人里帅的往往演技差"。
> **English**: DAG = a causal-structure graph (nodes=variables, arrows=direct causes). Three structures decide "control the middle variable or not": **confounder (X←Z→Y) must control** (else spurious correlation); **mediator (X→M→Y) don't control for the total effect** (blocks the path); **collider (X→C←Y) never control** (controlling conjures spurious correlation = selection bias). **Backdoor criterion**: control a set blocking all backdoor paths (through confounders) without touching mediators/colliders → unbiased causal effect. **do-operator**: $P(Y|do(X))$ (intervention, true causation) ≠ $P(Y|X)$ (observation, includes confounding). **Core lesson**: "more controls = more accurate" is wrong — controlling the wrong thing (mediator/collider) adds bias. Identifying confounders needs **domain knowledge to draw the DAG** (data alone can't tell direction). Collider examples: Berkson's paradox, survivorship bias, "why do famous good-looking people often act worse."


In [ ]:

# ============================================================
# 用模拟揭示三种结构 / reveal the three structures by simulation
# 中文:我们【知道】每种结构里真实的 X→Y 效应, 于是能验证"控制/不控制中间变量"哪个给出正确答案。
# English: we KNOW the true X→Y effect in each structure, so we can verify which of "control / don't control" is right.
# ============================================================
import numpy as np, matplotlib.pyplot as plt
import statsmodels.api as sm
rng=np.random.default_rng(0); N=5000
def ols_coef(y, *xs):                                        # X 对 Y 的回归系数(可加控制变量)/ regression coef of X
    Xmat=sm.add_constant(np.column_stack(xs)); return sm.OLS(y,Xmat).fit().params[1]

# ① 混杂 Confounder: Z→X, Z→Y, 真实 X→Y=0 / Z is a common cause, true X->Y = 0
Z=rng.normal(0,1,N); Xc=0.8*Z+rng.normal(0,1,N); Yc=0.8*Z+0.0*Xc+rng.normal(0,1,N)
conf_naive=ols_coef(Yc,Xc); conf_ctrl=ols_coef(Yc,Xc,Z)

# ② 中介 Mediator: X→M→Y, 真实总效应=0.9*0.5=0.45 / true TOTAL effect = 0.45
Xm=rng.normal(0,1,N); M=0.9*Xm+rng.normal(0,1,N); Ym=0.5*M+rng.normal(0,1,N)
med_total=ols_coef(Ym,Xm); med_ctrl=ols_coef(Ym,Xm,M)

# ③ 对撞 Collider: X→C←Y, X 与 Y 独立, 真实 X→Y=0 / X,Y independent, true = 0
Xo=rng.normal(0,1,N); Yo=rng.normal(0,1,N); C=0.9*Xo+0.9*Yo+rng.normal(0,0.3,N)
coll_naive=ols_coef(Yo,Xo); coll_ctrl=ols_coef(Yo,Xo,C)

print("① 混杂 Confounder (真实 X→Y = 0):")
print(f"   不控制 Z: {conf_naive:+.3f} (虚假相关!)   控制 Z: {conf_ctrl:+.3f} (≈0 正确) → 必须控制混杂")
print("② 中介 Mediator (真实总效应 = 0.45):")
print(f"   不控制 M: {med_total:+.3f} (总效应 正确)   控制 M: {med_ctrl:+.3f} (堵路→0 错误) → 求总效应别控中介")
print("③ 对撞 Collider (X,Y 独立, 真实 = 0):")
print(f"   不控制 C: {coll_naive:+.3f} (≈0 正确)   控制 C: {coll_ctrl:+.3f} (凭空造出强相关!) → 绝不能控对撞")


**中文**：三个结果一起看,颠覆了"控制变量越多越好"的直觉:
**English**: Seeing all three together overturns the intuition that "more controls is better":
- **混杂**:不控制 → 错(虚假相关);**控制才对**。
  **Confounder**: not controlling → wrong (spurious); **controlling is right**.
- **中介**:控制 → 错(堵住因果路);**不控制才对**(要总效应)。
  **Mediator**: controlling → wrong (blocks the path); **not controlling is right** (for the total effect).
- **对撞**:不控制 → 对;**控制反而错**(凭空造出相关)!
  **Collider**: not controlling → right; **controlling is wrong** (conjures correlation)!

**中文**：**同一个动作("控制某变量"),在三种结构里对错完全相反。** 所以你必须先画出 DAG、判断变量角色,才知道该控制谁。下面可视化三种 DAG 和对应的系数。
**English**: **The same action ("control a variable") is right or wrong depending on the structure.** So you must first draw the DAG and judge each variable's role to know what to control. Below we visualize the three DAGs and their coefficients.


In [ ]:

# ============================================================
# 可视化:三种 DAG + 控制/不控制的系数 / three DAGs + coefficients
# ============================================================
fig,axes=plt.subplots(2,3,figsize=(16,8))
def draw_dag(ax, edges, pos, title, highlight=None):
    for (u,v) in edges:
        ax.annotate("",xy=pos[v],xytext=pos[u],arrowprops=dict(arrowstyle="-|>",lw=2,color="#333",
                    shrinkA=15,shrinkB=15))
    for node,(x,y) in pos.items():
        c="#DD8452" if node==highlight else "#4C72B0"
        ax.scatter(x,y,s=1400,c=c,zorder=3,edgecolor="k")
        ax.text(x,y,node,ha="center",va="center",color="white",fontsize=13,fontweight="bold",zorder=4)
    ax.set_title(title,fontsize=11); ax.set_xlim(-0.5,2.5); ax.set_ylim(-0.5,1.5); ax.axis("off")
draw_dag(axes[0,0],[("Z","X"),("Z","Y")],{"X":(0,0),"Y":(2,0),"Z":(1,1)},"① 混杂 Confounder\nX←Z→Y (Z共同原因)",highlight="Z")
draw_dag(axes[0,1],[("X","M"),("M","Y")],{"X":(0,0),"M":(1,0.7),"Y":(2,0)},"② 中介 Mediator\nX→M→Y (M在路径上)",highlight="M")
draw_dag(axes[0,2],[("X","C"),("Y","C")],{"X":(0,0.7),"Y":(2,0.7),"C":(1,0)},"③ 对撞 Collider\nX→C←Y (C共同结果)",highlight="C")
# 下排:系数对比 / coefficient bars
specs=[("① 混杂",conf_naive,conf_ctrl,0.0,"控制Z"),("② 中介",med_total,med_ctrl,0.45,"控制M"),("③ 对撞",coll_naive,coll_ctrl,0.0,"控制C")]
for ax,(name,naive,ctrl,truth,lbl) in zip(axes[1],specs):
    ax.bar(["不控制\nnaive",lbl],[naive,ctrl],color=["#55A868","#C44E52"])
    ax.axhline(truth,ls="--",color="g",label=f"真实={truth}")
    ax.set_title(f"{name}: X→Y 系数"); ax.legend(fontsize=8)
    for i,v in enumerate([naive,ctrl]): ax.text(i,v,f"{v:+.2f}",ha="center",va="bottom" if v>=0 else "top",fontsize=9)
plt.tight_layout(); plt.savefig("/tmp/ci04_viz.png",dpi=80); plt.show()
print("绿=不控制, 红=控制。看绿红哪个贴近真实值(绿虚线)——三种结构答案完全不同!")


**中文**：诚实解读:
**English**: Honest takeaways:

**中文**：
1. **"控制变量越多越准"是彻底的误解**:三种结构里,"控制中间变量"这**同一个动作**分别是必须、不该、绝不能——**答案完全取决于因果结构**。盲目往回归里塞变量,轻则(控中介)低估效应,重则(控对撞)凭空造出根本不存在的强相关。这是因果推断与预测建模的**根本分野**:预测只要拟合好,因果必须结构对。
2. **对撞偏差是最反直觉、也最危险的**:$X,Y$ 明明独立,一旦条件在它们的共同结果 $C$ 上,就出现 −0.9 的强"相关"。现实中这以**选择偏差**的面目无处不在:①**伯克森悖论**——住院病人里"糖尿病"和"胆囊炎"负相关(因为得病才住院=条件在"住院"这个对撞上);②**幸存者偏差**——只看活下来的公司,会得出错误的成功因素;③"为什么明星里长得帅的演技差"——因为出名需要"帅**或**演技好"(条件在"出名"这个对撞上),于是在明星样本里两者负相关。**只要你的数据是被某种'结果'筛选过的,就要警惕对撞。**
3. **DAG 要靠领域知识画,数据画不出方向**:数据只能告诉你 $X,Z$ 相关,分不清是 $Z\to X$(混杂)还是 $X\to Z$(可能是中介或对撞)。**因果方向来自你对世界的理解**,不是从数据里跑出来的。这也是为什么因果推断永远需要**明确的假设**——DAG 就是把假设画出来、让人能审视和质疑。

**English**:
1. **"More controls = more accurate" is a complete misconception**: across the three structures, the **same action** "control the middle variable" is respectively mandatory, wrong, and forbidden — **the answer depends entirely on the causal structure**. Blindly stuffing variables into a regression at best (controlling a mediator) underestimates the effect, at worst (controlling a collider) conjures a strong correlation that doesn't exist. This is the **fundamental divide** between causal inference and predictive modeling: prediction just needs good fit; causation needs the right structure.
2. **Collider bias is the most counterintuitive and dangerous**: $X,Y$ are independent, yet conditioning on their common effect $C$ produces a strong −0.9 "correlation." In reality this pervades as **selection bias**: ① **Berkson's paradox** — among hospitalized patients, "diabetes" and "gallbladder disease" appear negatively correlated (because being sick is why you're hospitalized = conditioning on the "hospitalized" collider); ② **survivorship bias** — studying only surviving companies gives wrong success factors; ③ "why do good-looking celebrities act worse" — fame requires "looks **or** talent" (conditioning on the "famous" collider), so among celebrities the two are negatively correlated. **Whenever your data is filtered by some 'outcome,' beware colliders.**
3. **DAGs come from domain knowledge, not data**: data only tells you $X,Z$ are correlated; it can't tell $Z\to X$ (confounder) from $X\to Z$ (possibly a mediator or collider). **Causal direction comes from your understanding of the world**, not from running the data. This is why causal inference always needs **explicit assumptions** — the DAG draws those assumptions out for others to scrutinize and challenge.

> 💼 **实战视角 / Practical angle**
> **中文**:DAG 是做因果**前的第一步**:①**先画因果图**(和领域专家一起), 标出处理、结果、混杂、中介、对撞;②用**后门准则**确定要控制的变量集(工具:DoWhy 自动帮你找);③**只控混杂, 不碰中介/对撞**。常见错误:①"把能拿到的协变量全放进模型"(可能控了对撞/中介);②控制了**处理后变量**(post-treatment bias, 常是中介或对撞);③忽略了未观测混杂(这时才需要 19.7 IV / 19.8 RDD 等)。面试金句:*"因果推断先画 DAG:混杂必控(堵后门)、中介求总效应别控(否则堵路)、对撞绝不能控(否则选择偏差凭空造相关); do(X)≠观察X; 控制变量越多越准是错的——控错反而引入偏差。"*
> **English**: A DAG is **the first step before any causal analysis**: ① **draw the causal graph** (with domain experts), marking treatment, outcome, confounders, mediators, colliders; ② use the **backdoor criterion** to pick the control set (DoWhy automates this); ③ **control confounders only, never mediators/colliders**. Common mistakes: ① "put every available covariate in the model" (may control a collider/mediator); ② controlling a **post-treatment variable** (post-treatment bias, often a mediator or collider); ③ ignoring unobserved confounding (then you need IV 19.7 / RDD 19.8). Interview line: *"Causal inference starts by drawing the DAG: control confounders (block backdoors), don't control mediators for the total effect (blocks the path), never control colliders (selection bias conjures correlation); do(X) ≠ observing X; 'more controls = more accurate' is wrong — controlling the wrong variable adds bias."*

---
### 小结 / Summary
- **中文**:DAG=因果结构图; 三大结构:混杂(必控)、中介(求总效应别控)、对撞(绝不能控)。
- **English**: DAG = causal-structure graph; three structures: confounder (must control), mediator (don't control for total effect), collider (never control).
- **中文**:后门准则=控混杂堵后门、不碰中介/对撞; do(X)(干预)≠观察X(含混杂)。
- **English**: Backdoor criterion = control confounders to block backdoors, leave mediators/colliders alone; do(X) (intervention) ≠ observing X (confounded).
- **中文**:"控制越多越准"是错的; 对撞偏差最危险(选择偏差); DAG 靠领域知识画。
- **English**: "More controls = more accurate" is wrong; collider bias is most dangerous (selection bias); DAGs come from domain knowledge.
